In [1]:
%pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 765.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6

In [1]:
import json
from datasets import Dataset

with open("people_data.json","r") as f:
  data=json.load(f)

  tuning_data=[]

  for i in data:
    tuning_data.append(f"<|user|>\n{i["prompt"]}\n<|assistant|>\n{json.dumps(i["response"])}<|endoftext|>")

dataset=Dataset.from_dict({"text":tuning_data})

In [4]:
from unsloth import FastLanguageModel

model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.7.3: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:42: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
model=FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=64*2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

Unsloth 2026.7.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    max_seq_length=2048,
    tokenizer=tokenizer,
    dataset_text_field="text",
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=60,
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.805321
2,2.760300
3,2.623221
4,2.625227
5,2.637994
6,2.215807
7,2.225736
8,1.969483
9,1.840371
10,1.571963


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


TrainOutput(global_step=60, training_loss=0.9613177597522735, metrics={'train_runtime': 155.1404, 'train_samples_per_second': 3.094, 'train_steps_per_second': 0.387, 'total_flos': 775193131063296.0, 'train_loss': 0.9613177597522735, 'epoch': 1.5866666666666667})

In [15]:
FastLanguageModel.for_inference(model)

message=[
    {
      "role":"user",
      "content":"Mike is a 30 year old programmer.He loves Hiking"
}]

input=tokenizer.apply_chat_template(message, tokenizer=True, add_generation_prompt=True,return_tensors="pt").to("cuda")

output=model.generate(input_ids=input,max_new_tokens=512,use_cache=True,temperature=0.7,do_sample=True,top_p=0.9)

response=tokenizer.batch_decode(output)[0]

print(response)

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|> Mike is a 30 year old programmer.He loves Hiking<|end|><|assistant|> {"name": "Mike", "age": "30", "job": "programmer", "gender": "male"}<|end|>


In [16]:
model.save_pretrained_gguf("finetuned_model",tokenizer,quantization_method="q4_k_m",maximum_memory_usage=0.3)


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in finetuned_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in finetuned_model.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.65GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:20<00:00, 100.19s/it]


Unsloth: Merge process complete. Saved to `/content/finetuned_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10068-mix-fb3d4ca (app-b10068-mix-fb3d4ca-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['finetuned_model_gguf/phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complet

{'save_directory': 'finetuned_model',
 'gguf_directory': 'finetuned_model_gguf',
 'gguf_files': ['finetuned_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'finetuned_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}